# Dataset Consistency & Tension Analysis

Implements the two statistical estimators from **de Cruz Perez, Park & Ratra (2024)**, [arXiv:2404.19194](https://arxiv.org/abs/2404.19194), Section III.

---

## Required runs

You need **three completed COSMIX runs on the same model** to compare two datasets $D_1$ and $D_2$:

| Run | Likelihoods | Purpose |
|-----|-------------|---------|
| `run_D1` | $D_1$ only | individual DIC / logZ / posterior |
| `run_D2` | $D_2$ only | individual DIC / logZ / posterior |
| `run_D12` | $D_1 + D_2$ combined | joint DIC / logZ / posterior |

Dynesty runs are **required** for Estimator 2 (suspiciousness). Estimator 1 ($\log_{10}\mathcal{I}$) works with any sampler.

---

## Interpretation quick reference

### $\log_{10}\mathcal{I}$ (Estimator 1 — DIC-based)
$$\mathcal{G} = \text{DIC}(D_{12}) - \text{DIC}(D_1) - \text{DIC}(D_2), \qquad \log_{10}\mathcal{I} = -\mathcal{G}\,/\,(2\ln 10)$$
- $> 0$: datasets consistent; $< 0$: inconsistent  
- $|\log_{10}\mathcal{I}| > 0.5$ substantial · $> 1$ strong · $> 2$ decisive

### $p(\sigma)$ (Estimator 2 — Suspiciousness)
$$\ln S = \ln R - \ln I_{\rm ratio}, \qquad -2\ln S \sim \chi^2_d$$
$$p = P(\chi^2_d > -2\ln S), \qquad \sigma = \sqrt{2}\,\mathrm{erfc}^{-1}(1-p)$$
- $p \lesssim 0.05$ moderate tension ($\sim 2\sigma$) · $p \lesssim 0.003$ strong tension ($\sim 3\sigma$)

In [ ]:
import sys, os
# ── adjust if running the notebook from a directory other than COSMIX root ──
COSMIX_ROOT = os.path.abspath(".")
if COSMIX_ROOT not in sys.path:
    sys.path.insert(0, COSMIX_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from POST_PROCESSING_.DatasetConsistency import RunLoader, DatasetConsistency

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

## Configuration

Edit the paths and labels below to match your run directories.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDIT THESE PATHS
# ─────────────────────────────────────────────────────────────────────────────

# Run directories — relative to COSMIX_ROOT or absolute
PATH_D1  = "RUNS_/run_LCDM_CC_dynesty_1"       # D1-only run
PATH_D2  = "RUNS_/run_LCDM_PP_dynesty_1"        # D2-only run
PATH_D12 = "RUNS_/run_LCDM_CC_PP_dynesty_1"     # combined D1 + D2 run

# Human-readable labels
LABEL_D1  = "CC"
LABEL_D2  = "Pantheon+"
LABEL_D12 = "CC + Pantheon+"

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL: additional comparison pairs  (same model, different data splits)
# Uncomment and extend as needed; each entry is (path_D1, path_D2, path_D12, label1, label2)
# EXTRA_PAIRS = [
#     ("RUNS_/...", "RUNS_/...", "RUNS_/...", "BAO", "CMB"),
# ]
# ─────────────────────────────────────────────────────────────────────────────

## Load runs

In [ ]:
run1  = RunLoader.load(PATH_D1)
run2  = RunLoader.load(PATH_D2)
run12 = RunLoader.load(PATH_D12)

# ── Validation summary ───────────────────────────────────────────────────────
rows = []
for tag, lbl, run in [("D1", LABEL_D1, run1), ("D2", LABEL_D2, run2), ("D12", LABEL_D12, run12)]:
    rows.append({
        "Dataset"  : f"{tag}  ({lbl})",
        "Likelihoods" : ", ".join(run["likelihoods"]) or "(see manifest)",
        "N samples": len(run["chain"]),
        "Sampler"  : run["sampler_mode"],
        "Weights?" : "yes" if run["weights"] is not None else "NO (tension p unavailable)",
        "DIC"      : f"{run['DIC']:.3f}",
        "logZ_raw" : f"{run['logZ_raw']:.3f} ± {run['logZ_err']:.3f}" if run["logZ_raw"] is not None else "N/A",
    })

display(pd.DataFrame(rows).set_index("Dataset"))

---

## Estimator 1 — DIC-based $\log_{10}\mathcal{I}$

Works for **any** sampler.  Only needs DIC values from `diagnostics.json`.

In [ ]:
res1 = DatasetConsistency.log10_I(run1, run2, run12)

print(f"DIC({LABEL_D1})     = {run1['DIC']:.4f}")
print(f"DIC({LABEL_D2})     = {run2['DIC']:.4f}")
print(f"DIC({LABEL_D12})    = {run12['DIC']:.4f}")
print()
print(f"G  = DIC(D12) - DIC(D1) - DIC(D2) = {res1['G']:.4f}")
print(f"I  = exp(-G/2)                     = {res1['I']:.6f}")
print(f"log10(I)                            = {res1['log10_I']:+.4f}")
print()
print(f"Interpretation: {res1['label']}")

---

## Estimator 2 — Suspiciousness & tension probability $p(\sigma)$

Requires Dynesty runs with `weights.npy`.  
Based on Handley & Lemos (2019b), [arXiv:1902.04029](https://arxiv.org/abs/1902.04029).

In [ ]:
try:
    res2 = DatasetConsistency.suspiciousness_and_tension(run1, run2, run12)

    if res2["warnings"]:
        print("⚠  Warnings:")
        for w in res2["warnings"]:
            print(f"   {w}")
        print()

    print("── KL divergences & Bayesian model complexity ────────────────────")
    print(f"  D_KL({LABEL_D1})     = {res2['D_KL_1']:.4f}   d̃ = {res2['d_tilde_1']:.4f}")
    print(f"  D_KL({LABEL_D2})     = {res2['D_KL_2']:.4f}   d̃ = {res2['d_tilde_2']:.4f}")
    print(f"  D_KL({LABEL_D12})    = {res2['D_KL_12']:.4f}   d̃ = {res2['d_tilde_12']:.4f}")
    print(f"  Bayesian model dimensionality d = {res2['d']:.3f}")
    print(f"  (Expected ≈ {len(run12['param_names'])} free parameters)")
    print()
    print("── Tension estimators ────────────────────────────────────────────")
    print(f"  ln R  (Bayes ratio)       = {res2['ln_R']:+.4f}")
    print(f"  ln I  (information ratio) = {res2['ln_I']:+.4f}")
    print(f"  ln S  (suspiciousness)    = {res2['ln_S']:+.4f}")
    print(f"  -2 ln S                   = {res2['chi2_tension']:.4f}")
    print()
    print(f"  p (tension probability)   = {res2['p']:.4f}  ({100*res2['p']:.2f}%)")
    print(f"  σ (Gaussian equivalent)   = {res2['sigma']:.3f}σ  ±  {res2['sigma_logZ_err']:.3f} (logZ uncertainty)")
    print()
    print(f"  Interpretation: {res2['label']}")

except (ValueError, KeyError) as e:
    print(f"Cannot compute suspiciousness: {e}")

---

## Summary Table

In [ ]:
summary = {
    "Comparison"        : f"{LABEL_D1} vs {LABEL_D2}",
    "G"                 : f"{res1['G']:.4f}",
    "log10(I)"          : f"{res1['log10_I']:+.4f}",
    "Consistency (DIC)" : res1["label"],
}

if "res2" in dir() and not np.isnan(res2.get("sigma", float("nan"))):
    summary.update({
        "d  (Bayes dim)"   : f"{res2['d']:.2f}",
        "ln S"             : f"{res2['ln_S']:+.4f}",
        "p"                : f"{res2['p']:.4f}",
        "σ"                : f"{res2['sigma']:.3f} ± {res2['sigma_logZ_err']:.3f}",
        "Tension (susp.)"  : res2["label"],
    })

df_summary = pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])
display(df_summary)

---

## Visualisation — logZ uncertainty & tension level

In [ ]:
if "res2" in dir() and not np.isnan(res2.get("sigma", float("nan"))):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # ── Left: sigma vs log10(I) gauge ──────────────────────────────────────
    ax = axes[0]
    sigma_val = res2["sigma"]
    sigma_err = res2["sigma_logZ_err"]
    log10I    = res1["log10_I"]

    # Reference thresholds
    ax.axvline(0,    color="gray",  lw=1, ls="--")
    ax.axvline(-0.5, color="gold",  lw=1, ls=":",  label="substantial")
    ax.axvline(-1.0, color="orange",lw=1, ls=":",  label="strong")
    ax.axvline(-2.0, color="red",   lw=1, ls=":",  label="decisive")
    ax.axvline(+0.5, color="gold",  lw=1, ls=":")
    ax.axvline(+1.0, color="orange",lw=1, ls=":")
    ax.axvline(+2.0, color="red",   lw=1, ls=":")

    ax.axvspan(-0.5,  0.5, alpha=0.07, color="green")
    ax.axvspan(-1.0, -0.5, alpha=0.07, color="yellow")
    ax.axvspan(-2.0, -1.0, alpha=0.07, color="orange")
    ax.axvspan(-3.5, -2.0, alpha=0.07, color="red")
    ax.axvspan( 0.5,  1.0, alpha=0.07, color="yellow")
    ax.axvspan( 1.0,  2.0, alpha=0.07, color="orange")
    ax.axvspan( 2.0,  3.5, alpha=0.07, color="royalblue")

    ax.axhline(0, color="k", lw=0.5)
    ax.plot(log10I, 0, "D", color="steelblue", ms=12, zorder=5,
            label=f"log₁₀(I) = {log10I:+.3f}")
    ax.set_xlabel(r"$\log_{10}\,\mathcal{I}$", fontsize=13)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-1, 1)
    ax.yaxis.set_visible(False)
    ax.set_title(f"DIC-based consistency\n{LABEL_D1} vs {LABEL_D2}", fontsize=11)
    ax.legend(fontsize=9, loc="lower right")

    # ── Right: tension sigma ───────────────────────────────────────────────
    ax = axes[1]
    ax.axvline(1, color="green",  lw=1, ls="--", label="1σ")
    ax.axvline(2, color="gold",   lw=1, ls="--", label="2σ (moderate)")
    ax.axvline(3, color="red",    lw=1, ls="--", label="3σ (strong)")
    ax.axvspan(0, 1, alpha=0.08, color="green")
    ax.axvspan(1, 2, alpha=0.08, color="yellow")
    ax.axvspan(2, 3, alpha=0.08, color="orange")
    ax.axvspan(3, 6, alpha=0.08, color="red")

    ax.axhline(0, color="k", lw=0.5)
    ax.errorbar(sigma_val, 0, xerr=sigma_err, fmt="D", color="crimson", ms=12,
                capsize=6, lw=2, zorder=5,
                label=f"σ = {sigma_val:.2f} ± {sigma_err:.2f}")
    ax.set_xlabel(r"Tension $\sigma$", fontsize=13)
    ax.set_xlim(0, max(6, sigma_val + 1.5))
    ax.set_ylim(-1, 1)
    ax.yaxis.set_visible(False)
    ax.set_title(f"Suspiciousness tension\n{LABEL_D1} vs {LABEL_D2}", fontsize=11)
    ax.legend(fontsize=9, loc="upper right")

    fig.suptitle(f"Dataset Consistency: {LABEL_D1} vs {LABEL_D2}", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(f"tension_{LABEL_D1}_vs_{LABEL_D2}.pdf", bbox_inches="tight")
    plt.show()
else:
    print("Visualisation requires a successful suspiciousness calculation.")

---

## Multiple pairs (optional)

Uncomment and populate `EXTRA_PAIRS` in the config cell to run multiple comparisons in a single table.

In [ ]:
# Run this cell only if EXTRA_PAIRS is defined in the config cell.
if "EXTRA_PAIRS" in dir():
    all_rows = []
    for p1, p2, p12, l1, l2 in EXTRA_PAIRS:
        r1  = RunLoader.load(p1)
        r2  = RunLoader.load(p2)
        r12 = RunLoader.load(p12)
        e1  = DatasetConsistency.log10_I(r1, r2, r12)
        row = {"Comparison": f"{l1} vs {l2}",
               "log10(I)": f"{e1['log10_I']:+.4f}",
               "Consistency": e1["label"]}
        try:
            e2 = DatasetConsistency.suspiciousness_and_tension(r1, r2, r12)
            row["σ"]       = f"{e2['sigma']:.3f} ± {e2['sigma_logZ_err']:.3f}"
            row["p"]       = f"{e2['p']:.4f}"
            row["Tension"] = e2["label"]
        except Exception as exc:
            row["σ"] = row["p"] = row["Tension"] = f"N/A ({exc})"
        all_rows.append(row)

    display(pd.DataFrame(all_rows).set_index("Comparison"))
else:
    print("No extra pairs configured.")

---

## Sanity checks

These cells help verify that the runs were set up correctly before interpreting the results.

In [ ]:
# Bayesian model dimensionality check
# d should be approximately equal to the number of free parameters.
# Large deviations indicate prior-dominated parameters or mismatched runs.
if "res2" in dir():
    n_free = len(run12["param_names"])
    d_val  = res2["d"]
    print(f"Free parameters (from manifest) : {n_free}")
    print(f"Bayesian model dimensionality d : {d_val:.3f}")
    ratio = d_val / n_free if n_free > 0 else float("nan")
    print(f"Ratio d / n_free                : {ratio:.3f}  (healthy range: 0.5 – 1.5)")
    if ratio < 0.3 or ratio > 2.0:
        print("⚠  d is far from n_free. Possible causes:")
        print("   • Runs used different models or parameter spaces")
        print("   • Posterior very prior-dominated (nearly unconstrained parameters)")
        print("   • Extremely low ESS (poor sampling)")
    else:
        print("✓  d looks healthy.")

# logZ consistency: logZ_D1 + logZ_D2 vs logZ_D12
# Under perfect consistency (same posteriors), logZ_D12 ≈ logZ_D1 + logZ_D2
# Large negative difference indicates tension.
if all(r["logZ_raw"] is not None for r in [run1, run2, run12]):
    lnR = run12["logZ_raw"] - run1["logZ_raw"] - run2["logZ_raw"]
    print(f"\nln R = ln Z_12 - ln Z_1 - ln Z_2 = {lnR:+.4f}")
    print("  (> 0: Bayesian evidence favors joint; < 0: penalised by conflict)")

In [ ]:
# Shannon information plot — check that the KL-divergence estimator is well-sampled.
# The posterior histogram of (log L - log Z) should look like a roughly Gaussian
# distribution shifted right of zero. Spiky or multimodal histograms suggest
# insufficient ESS and unreliable d̃ estimates.
if "res2" in dir():
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)
    for ax, run, lbl, dkl, dtilde in [
        (axes[0], run1,  LABEL_D1,  res2["D_KL_1"],  res2["d_tilde_1"]),
        (axes[1], run2,  LABEL_D2,  res2["D_KL_2"],  res2["d_tilde_2"]),
        (axes[2], run12, LABEL_D12, res2["D_KL_12"], res2["d_tilde_12"]),
    ]:
        logl  = run["log_prob"]
        logZ  = run["logZ_raw"]
        w     = run["weights"]
        if w is None:
            w = np.ones(len(logl)) / len(logl)
        log_IS = logl - logZ
        ax.hist(log_IS, bins=50, weights=w, color="steelblue", alpha=0.7, density=True)
        ax.axvline(dkl, color="red", lw=2, label=f"D_KL = {dkl:.2f}")
        ax.set_xlabel(r"$\ln\mathcal{L} - \ln Z$ (Shannon info)", fontsize=10)
        ax.set_title(f"{lbl}\nd̃ = {dtilde:.2f}", fontsize=10)
        ax.legend(fontsize=9)
    fig.suptitle("Shannon information distributions (sanity check)", fontsize=12)
    plt.tight_layout()
    plt.show()